## 1. Imports & Setup

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — required for PDF saving
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Flatten,
    Dropout, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2

print("TF version:", tf.__version__)
print("Libraries imported successfully")

E0000 00:00:1780748291.945165      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780748292.057785      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780748293.019118      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780748293.019178      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780748293.019181      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780748293.019184      58 computation_placer.cc:177] computation placer already registered. Please check linka

TF version: 2.19.0
Libraries imported successfully


## 2. Output Directories

In [2]:
OUTPUT_DIR  = "/kaggle/working/outputs"
CSV_DIR     = os.path.join(OUTPUT_DIR, "csv")
MODEL_DIR   = os.path.join(OUTPUT_DIR, "models")

for d in [OUTPUT_DIR, CSV_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

PDF_PATH = os.path.join(OUTPUT_DIR, "Traffic_Sign_Recognition_Results.pdf")
print("Output dirs ready")
print("All figures → PDF:", PDF_PATH)
print("All tables  → CSV:", CSV_DIR)

Output dirs ready
All figures → PDF: /kaggle/working/outputs/Traffic_Sign_Recognition_Results.pdf
All tables  → CSV: /kaggle/working/outputs/csv


## 3. Global PdfPages Context

We open ONE `PdfPages` object and append every figure to it throughout the notebook.
At the very end one call to `pdf.close()` finalises the file.

In [3]:
# Open the PDF writer — keep it alive for the whole notebook
pdf = PdfPages(PDF_PATH)

def save_fig_to_pdf(fig, title=""):
    """Adds a figure to the running PDF and closes it to free memory."""
    if title:
        fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)
    print(f"  → saved '{title}' to PDF")

print("PdfPages context open")

PdfPages context open


## 4. Class Name Mapping

In [4]:
CLASS_NAMES = {
    0:  "Speed limit 20",       1:  "Speed limit 30",
    2:  "Speed limit 50",       3:  "Speed limit 60",
    4:  "Speed limit 70",       5:  "Speed limit 80",
    6:  "End speed limit 80",   7:  "Speed limit 100",
    8:  "Speed limit 120",      9:  "No passing",
    10: "No passing >3.5t",     11: "Right-of-way",
    12: "Priority road",        13: "Yield",
    14: "Stop",                 15: "No vehicles",
    16: "No vehicles >3.5t",    17: "No entry",
    18: "General caution",      19: "Curve left",
    20: "Curve right",          21: "Double curve",
    22: "Bumpy road",           23: "Slippery road",
    24: "Road narrows right",   25: "Road work",
    26: "Traffic signals",      27: "Pedestrians",
    28: "Children crossing",    29: "Bicycles crossing",
    30: "Ice/snow",             31: "Wild animals",
    32: "End restrictions",     33: "Turn right ahead",
    34: "Turn left ahead",      35: "Go ahead",
    36: "Go ahead or right",    37: "Go ahead or left",
    38: "Keep right",           39: "Keep left",
    40: "Roundabout",           41: "End no passing",
    42: "End no passing >3.5t"
}
print("43 class names loaded")

43 class names loaded


## 5. Load & Pre-process Data

In [5]:
dataset_path = "/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"
train_path   = os.path.join(dataset_path, "Train")

MAX_PER_CLASS = 300
IMG_SIZE      = 32

data, labels = [], []

for class_id in range(43):
    class_folder = os.path.join(train_path, str(class_id))
    if not os.path.exists(class_folder):
        continue
    for img_name in os.listdir(class_folder)[:MAX_PER_CLASS]:
        img_path = os.path.join(class_folder, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        data.append(image)
        labels.append(class_id)

data   = np.array(data,   dtype="float32") / 255.0
labels = np.array(labels, dtype="int32")

print("Data shape  :", data.shape)
print("Labels shape:", labels.shape)
print("Pixel range :", data.min(), "–", data.max())

Data shape  : (12330, 32, 32, 3)
Labels shape: (12330,)
Pixel range : 0.0 – 1.0


## 6. Train / Val / Test Split  (stratified on raw labels, one-hot AFTER split)

In [6]:
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    data, labels, test_size=0.30, random_state=42, stratify=labels
)
X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.50, random_state=42, stratify=y_temp_raw
)

y_train = to_categorical(y_train_raw, 43)
y_val   = to_categorical(y_val_raw,   43)
y_test  = to_categorical(y_test_raw,  43)

print(f"Train : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}")

# Save split summary as CSV
split_df = pd.DataFrame({
    "Split" : ["Train", "Validation", "Test"],
    "Samples": [len(X_train), len(X_val), len(X_test)]
})
split_df.to_csv(os.path.join(CSV_DIR, "dataset_split.csv"), index=False)
print("Saved dataset_split.csv")

Train : (8631, 32, 32, 3)  |  Val : (1849, 32, 32, 3)  |  Test : (1850, 32, 32, 3)
Saved dataset_split.csv


## 7. Class Distribution Chart

In [7]:
unique, counts = np.unique(y_train_raw, return_counts=True)

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(unique, counts, color='steelblue', edgecolor='white')
ax.set_xlabel("Class ID")
ax.set_ylabel("Number of Samples")
ax.set_xticks(unique)
ax.set_xticklabels(
    [f"{i}" for i in unique], rotation=90, fontsize=7
)
ax.grid(axis='y', alpha=0.3)
save_fig_to_pdf(fig, "Class Distribution — Training Set")

# Also save as CSV
dist_df = pd.DataFrame({
    "Class_ID" : unique,
    "Class_Name": [CLASS_NAMES[i] for i in unique],
    "Count"    : counts
})
dist_df.to_csv(os.path.join(CSV_DIR, "class_distribution.csv"), index=False)
print("Saved class_distribution.csv")

  → saved 'Class Distribution — Training Set' to PDF
Saved class_distribution.csv


## 8. Sample Image Grid

In [8]:
fig, axes = plt.subplots(5, 9, figsize=(16, 10))
axes = axes.flatten()

for cls_id in range(43):
    idx = np.where(y_train_raw == cls_id)[0]
    if len(idx) == 0:
        axes[cls_id].axis('off')
        continue
    sample = X_train[idx[0]]
    axes[cls_id].imshow(sample)
    axes[cls_id].set_title(f"{cls_id}\n{CLASS_NAMES[cls_id][:12]}",
                            fontsize=5.5, pad=2)
    axes[cls_id].axis('off')

# Hide extra axes
for i in range(43, len(axes)):
    axes[i].axis('off')

save_fig_to_pdf(fig, "Sample Images — One per Class")

  → saved 'Sample Images — One per Class' to PDF


## 9. CNN Model

In [9]:
cnn_model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),
    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    # Head
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(43, activation='softmax')
], name="CNN")

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 43)             │        11,051 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 629,739 (2.40 MB)

 Trainable params: 629,291 (2.40 MB)

 Non-trainable params: 448 (1.75 KB)

In [10]:
cnn_history = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 17s 102ms/step - accuracy: 0.2406 - loss: 2.9203 - val_accuracy: 0.0243 - val_loss: 6.3367
Epoch 2/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 13s 98ms/step - accuracy: 0.6374 - loss: 1.1865 - val_accuracy: 0.1174 - val_loss: 4.2208
Epoch 3/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.8334 - loss: 0.5092 - val_accuracy: 0.5425 - val_loss: 1.5484
Epoch 4/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.9110 - loss: 0.2705 - val_accuracy: 0.9286 - val_loss: 0.2403
Epoch 5/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 13s 98ms/step - accuracy: 0.9486 - loss: 0.1615 - val_accuracy: 0.9492 - val_loss: 0.1547
Epoch 6/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 14s 103ms/step - accuracy: 0.9567 - loss: 0.1327 - val_accuracy: 0.9605 - val_loss: 0.1180
Epoch 7/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 13s 98ms/step - accuracy: 0.9674 - loss: 0.1013 - val_accuracy: 0.9659 - val_loss: 0.1058
Epoch 8/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 13s 98ms/step - accuracy: 0.9749 - loss: 0.0832 

## 10. MobileNetV2 Model

In [11]:
MN_SIZE = 96

# Resize first, THEN preprocess (correct order)
X_train_mn = tf.keras.applications.mobilenet_v2.preprocess_input(
    tf.cast(tf.image.resize(X_train, (MN_SIZE, MN_SIZE)), tf.float32)
).numpy()
X_val_mn   = tf.keras.applications.mobilenet_v2.preprocess_input(
    tf.cast(tf.image.resize(X_val,   (MN_SIZE, MN_SIZE)), tf.float32)
).numpy()
X_test_mn  = tf.keras.applications.mobilenet_v2.preprocess_input(
    tf.cast(tf.image.resize(X_test,  (MN_SIZE, MN_SIZE)), tf.float32)
).numpy()

print("MobileNet inputs ready:", X_train_mn.shape)

MobileNet inputs ready: (8631, 96, 96, 3)


In [12]:
base_model = MobileNetV2(
    weights=None, include_top=False, input_shape=(MN_SIZE, MN_SIZE, 3)
)
base_model.trainable = True

x      = GlobalAveragePooling2D()(base_model.output)
output = Dense(43, activation='softmax')(x)

mobilenet_model = Model(inputs=base_model.input, outputs=output, name="MobileNetV2")
mobilenet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("MobileNetV2 model built")

MobileNetV2 model built


In [13]:
mobilenet_history = mobilenet_model.fit(
    X_train_mn, y_train,
    validation_data=(X_val_mn, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 208s 1s/step - accuracy: 0.2798 - loss: 2.5779 - val_accuracy: 0.0243 - val_loss: 3.7646
Epoch 2/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 177s 1s/step - accuracy: 0.7723 - loss: 0.7161 - val_accuracy: 0.0243 - val_loss: 3.8099
Epoch 3/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 173s 1s/step - accuracy: 0.8945 - loss: 0.3169 - val_accuracy: 0.0243 - val_loss: 3.9029
Epoch 4/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 172s 1s/step - accuracy: 0.9392 - loss: 0.1925 - val_accuracy: 0.0243 - val_loss: 4.0863
Epoch 5/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 176s 1s/step - accuracy: 0.9477 - loss: 0.1691 - val_accuracy: 0.0243 - val_loss: 4.3495
Epoch 6/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 169s 1s/step - accuracy: 0.9614 - loss: 0.1191 - val_accuracy: 0.0243 - val_loss: 4.7227
Epoch 7/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 171s 1s/step - accuracy: 0.9626 - loss: 0.1175 - val_accuracy: 0.0243 - val_loss: 5.1090
Epoch 8/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 171s 1s/step - accuracy: 0.9795 - loss: 0.0666 - val_accu

## 11. Training Curves → PDF

In [14]:
def plot_training_curves(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, metric, ylabel in zip(
        axes,
        ['accuracy', 'loss'],
        ['Accuracy', 'Loss']
    ):
        ax.plot(history.history[metric],      label='Train',      linewidth=2)
        ax.plot(history.history[f'val_{metric}'], label='Validation', linewidth=2, linestyle='--')
        ax.set_title(f"{model_name} — {ylabel}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(alpha=0.3)

    save_fig_to_pdf(fig, f"{model_name} Training Curves")

    # Save history as CSV
    hist_df = pd.DataFrame(history.history)
    hist_df.index.name = 'epoch'
    csv_name = f"{model_name.lower().replace(' ','_')}_training_history.csv"
    hist_df.to_csv(os.path.join(CSV_DIR, csv_name))
    print(f"Saved {csv_name}")

plot_training_curves(cnn_history,       "CNN")
plot_training_curves(mobilenet_history, "MobileNetV2")

  → saved 'CNN Training Curves' to PDF
Saved cnn_training_history.csv
  → saved 'MobileNetV2 Training Curves' to PDF
Saved mobilenetv2_training_history.csv


## 12. Evaluation — Classification Reports → PDF + CSV

In [15]:
def evaluate_model(model, X, y_true_raw, X_label="Test", model_name="Model"):
    """Run predictions, save classification report as CSV, confusion matrix to PDF."""
    y_pred_raw = np.argmax(model.predict(X, verbose=0), axis=1)
    acc = accuracy_score(y_true_raw, y_pred_raw)
    print(f"{model_name} [{X_label}] Accuracy: {acc:.4f}")

    # --- Classification report → CSV ---
    report_dict = classification_report(
        y_true_raw, y_pred_raw,
        target_names=[CLASS_NAMES[i] for i in range(43)],
        output_dict=True, zero_division=0
    )
    report_df = pd.DataFrame(report_dict).T.round(4)
    csv_name  = f"{model_name.lower().replace(' ','_')}_classification_report.csv"
    report_df.to_csv(os.path.join(CSV_DIR, csv_name))
    print(f"  Saved {csv_name}")

    # --- Confusion matrix → PDF ---
    cm  = confusion_matrix(y_true_raw, y_pred_raw)
    fig = plt.figure(figsize=(14, 12))
    sns.heatmap(
        cm, annot=False, cmap="Blues",
        xticklabels=range(43), yticklabels=range(43),
        linewidths=0
    )
    plt.xlabel("Predicted Class", fontsize=11)
    plt.ylabel("True Class",      fontsize=11)
    save_fig_to_pdf(fig, f"{model_name} — Confusion Matrix ({X_label} Set, Acc={acc:.3f})")

    # --- Per-class bar chart → PDF ---
    per_class = np.diag(cm) / (cm.sum(axis=1) + 1e-9)
    fig2, ax2 = plt.subplots(figsize=(16, 5))
    colors = ['#2ecc71' if v >= 0.8 else '#e67e22' if v >= 0.5 else '#e74c3c'
               for v in per_class]
    ax2.bar(range(43), per_class, color=colors)
    ax2.set_xticks(range(43))
    ax2.set_xticklabels(
        [f"{i}" for i in range(43)], rotation=90, fontsize=7
    )
    ax2.set_ylim(0, 1.05)
    ax2.axhline(0.8, color='green',  linestyle='--', alpha=0.6, label='80% threshold')
    ax2.axhline(0.5, color='orange', linestyle='--', alpha=0.6, label='50% threshold')
    ax2.set_xlabel("Class ID")
    ax2.set_ylabel("Per-Class Accuracy")
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    save_fig_to_pdf(fig2, f"{model_name} — Per-Class Accuracy ({X_label} Set)")

    # Save per-class accuracy CSV
    pca_df = pd.DataFrame({
        "Class_ID"  : range(43),
        "Class_Name": [CLASS_NAMES[i] for i in range(43)],
        "Accuracy"  : per_class.round(4)
    })
    pca_csv = f"{model_name.lower().replace(' ','_')}_per_class_accuracy.csv"
    pca_df.to_csv(os.path.join(CSV_DIR, pca_csv), index=False)
    print(f"  Saved {pca_csv}")

    return y_pred_raw, acc

cnn_preds,  cnn_acc  = evaluate_model(cnn_model,       X_test,    y_test_raw, model_name="CNN")
mn_preds,   mn_acc   = evaluate_model(mobilenet_model, X_test_mn, y_test_raw, model_name="MobileNetV2")

CNN [Test] Accuracy: 0.9724
  Saved cnn_classification_report.csv
  → saved 'CNN — Confusion Matrix (Test Set, Acc=0.972)' to PDF
  → saved 'CNN — Per-Class Accuracy (Test Set)' to PDF
  Saved cnn_per_class_accuracy.csv
MobileNetV2 [Test] Accuracy: 0.0243
  Saved mobilenetv2_classification_report.csv
  → saved 'MobileNetV2 — Confusion Matrix (Test Set, Acc=0.024)' to PDF
  → saved 'MobileNetV2 — Per-Class Accuracy (Test Set)' to PDF
  Saved mobilenetv2_per_class_accuracy.csv


## 13. Model Comparison Table → PDF + CSV

In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

comp = pd.DataFrame({
    "Model"    : ["CNN", "MobileNetV2"],
    "Accuracy" : [cnn_acc, mn_acc],
    "Precision": [
        precision_score(y_test_raw, cnn_preds, average='macro', zero_division=0),
        precision_score(y_test_raw, mn_preds,  average='macro', zero_division=0)
    ],
    "Recall"   : [
        recall_score(y_test_raw, cnn_preds, average='macro', zero_division=0),
        recall_score(y_test_raw, mn_preds,  average='macro', zero_division=0)
    ],
    "F1-Score" : [
        f1_score(y_test_raw, cnn_preds, average='macro', zero_division=0),
        f1_score(y_test_raw, mn_preds,  average='macro', zero_division=0)
    ]
}).round(4)

comp.to_csv(os.path.join(CSV_DIR, "model_comparison.csv"), index=False)
print(comp.to_string(index=False))

# Render table as PDF page
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.axis('off')
table = ax.table(
    cellText   = comp.values,
    colLabels  = comp.columns,
    loc        = 'center',
    cellLoc    = 'center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)

# Highlight header
for j in range(len(comp.columns)):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

save_fig_to_pdf(fig, "Model Comparison — Test Set Metrics")
print("Saved model_comparison.csv")

      Model  Accuracy  Precision  Recall  F1-Score
        CNN    0.9724     0.9744  0.9734    0.9735
MobileNetV2    0.0243     0.0006  0.0233    0.0011
  → saved 'Model Comparison — Test Set Metrics' to PDF
Saved model_comparison.csv


## 14. Grad-CAM — Clearly Labeled per Class

In [17]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """
    Returns a (H, W) heatmap in [0,1] for the predicted class.
    Fixes:
      - conv_outputs is explicitly watched by the tape
      - pred_class is cast to plain Python int before indexing
      - handles scalar heatmap edge-case (single-pixel conv output)
    """
    grad_model = Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output,
                   model.output]
    )

    img_tensor = tf.cast(np.expand_dims(img_array, 0), tf.float32)

    with tf.GradientTape() as tape:
        # Explicitly watch the conv output so gradients flow
        conv_outputs, predictions = grad_model(img_tensor, training=False)
        tape.watch(conv_outputs)
        # Use plain int index — avoids EagerTensor indexing issues
        pred_class = int(tf.argmax(predictions[0]).numpy())
        loss = predictions[:, pred_class]

    grads  = tape.gradient(loss, conv_outputs)   # (1, h, w, filters)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))  # (filters,)

    # Weighted combination of conv feature maps
    heatmap = tf.reduce_sum(
        conv_outputs[0] * pooled, axis=-1
    ).numpy()  # (h, w)

    heatmap = np.maximum(heatmap, 0)  # ReLU
    denom   = heatmap.max() + 1e-9
    return (heatmap / denom).astype(np.float32)


def overlay_gradcam(img_norm, heatmap, alpha=0.45):
    """
    Superimposes a Grad-CAM heatmap on a normalised [0,1] RGB image.
    Returns a uint8 RGB array.
    """
    h, w = img_norm.shape[:2]
    # Ensure float32 before resize
    hm = heatmap.astype(np.float32)
    hm_resized = cv2.resize(hm, (w, h), interpolation=cv2.INTER_LINEAR)
    hm_colored = cv2.applyColorMap(
        np.uint8(255 * hm_resized), cv2.COLORMAP_JET
    )
    hm_colored = cv2.cvtColor(hm_colored, cv2.COLOR_BGR2RGB)
    img_uint8  = np.uint8(np.clip(img_norm * 255, 0, 255))
    overlay    = cv2.addWeighted(img_uint8, 1 - alpha, hm_colored, alpha, 0)
    return overlay


# Find last conv layer name in CNN
last_conv = [l.name for l in cnn_model.layers if 'conv' in l.name.lower()][-1]
print("Last conv layer used for Grad-CAM:", last_conv)

Last conv layer used for Grad-CAM: conv2d_2


In [18]:
# --- Grad-CAM grid: 2 rows (original + overlay), 9 classes per PDF page ---

CLASSES_TO_SHOW = list(range(43))
IMAGES_PER_ROW  = 9

pages = [
    CLASSES_TO_SHOW[i : i + IMAGES_PER_ROW]
    for i in range(0, len(CLASSES_TO_SHOW), IMAGES_PER_ROW)
]

for page_idx, cls_batch in enumerate(pages):
    n = len(cls_batch)

    # Always force axes to be 2-D array (2 rows × n cols)
    fig, axes = plt.subplots(2, n, figsize=(n * 2.3, 5.8))
    axes = np.array(axes)          # ensure ndarray
    if axes.ndim == 1:             # n==1 gives shape (2,) — fix to (2,1)
        axes = axes.reshape(2, 1)

    for col, cls_id in enumerate(cls_batch):
        # --- hide both axes first; fill only if data is available ---
        axes[0, col].axis('off')
        axes[1, col].axis('off')

        idxs = np.where(y_test_raw == cls_id)[0]
        if len(idxs) == 0:
            axes[0, col].set_title(f"Class {cls_id}\n(no samples)",
                                    fontsize=6.5)
            continue

        img = X_test[idxs[0]]  # (32,32,3) float32 in [0,1]

        try:
            heatmap = make_gradcam_heatmap(img, cnn_model, last_conv)
            overlay = overlay_gradcam(img, heatmap)
        except Exception as e:
            print(f"  Grad-CAM failed for class {cls_id}: {e}")
            axes[0, col].set_title(f"Class {cls_id}\nError", fontsize=6)
            continue

        # Row 0 — original image with class label
        pred_cls = int(np.argmax(cnn_model.predict(img[np.newaxis], verbose=0)[0]))
        correct  = "✓" if pred_cls == cls_id else "✗"
        axes[0, col].imshow(img)
        axes[0, col].set_title(
            f"{correct} C{cls_id}\n{CLASS_NAMES[cls_id][:14]}",
            fontsize=6.5, pad=2,
            color='green' if pred_cls == cls_id else 'red'
        )
        axes[0, col].axis('off')

        # Row 1 — Grad-CAM overlay
        axes[1, col].imshow(overlay)
        axes[1, col].set_title(
            f"Pred: C{pred_cls}\n{CLASS_NAMES[pred_cls][:14]}",
            fontsize=6, pad=2
        )
        axes[1, col].axis('off')

    # Row labels on the leftmost column
    axes[0, 0].set_ylabel("Original",  fontsize=8, labelpad=4)
    axes[1, 0].set_ylabel("Grad-CAM",  fontsize=8, labelpad=4)

    fig.subplots_adjust(wspace=0.05, hspace=0.45)

    save_fig_to_pdf(
        fig,
        f"CNN Grad-CAM  |  Classes {cls_batch[0]}–{cls_batch[-1]}  "
        f"(page {page_idx + 1} / {len(pages)})"
    )

print("All Grad-CAM pages saved to PDF")

  Grad-CAM failed for class 0: The layer CNN has never been called and thus has no defined output.
  Grad-CAM failed for class 1: The layer CNN has never been called and thus has no defined output.
  Grad-CAM failed for class 2: The layer CNN has never been called and thus has no defined output.
  Grad-CAM failed for class 3: The layer CNN has never been called and thus has no defined output.
  Grad-CAM failed for class 4: The layer CNN has never been called and thus has no defined output.
  Grad-CAM failed for class 5: The layer CNN has never been called and thus has no defined output.
  Grad-CAM failed for class 6: The layer CNN has never been called and thus has no defined output.
  Grad-CAM failed for class 7: The layer CNN has never been called and thus has no defined output.
  Grad-CAM failed for class 8: The layer CNN has never been called and thus has no defined output.
  → saved 'CNN Grad-CAM  |  Classes 0–8  (page 1 / 5)' to PDF
  Grad-CAM failed for class 9: The layer CNN ha

## 15. SHAP Explanations

In [19]:
try:
    import shap
    print("SHAP version:", shap.__version__)
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
    import shap
    print("SHAP installed and imported")

SHAP version: 0.51.0


In [20]:
# --- SHAP GradientExplainer (works well with CNNs) ---

# Background: small random subset of training images
np.random.seed(42)
bg_idx  = np.random.choice(len(X_train), size=100, replace=False)
bg_data = X_train[bg_idx]

explainer = shap.GradientExplainer(cnn_model, bg_data)
print("SHAP GradientExplainer ready")

# Explain one test sample per class (first 9 classes → one PDF page)
SHAP_CLASSES = list(range(9))   # increase to 43 for full run (slower)
shap_images, shap_vals, shap_labels = [], [], []

for cls_id in SHAP_CLASSES:
    idxs = np.where(y_test_raw == cls_id)[0]
    if len(idxs) == 0:
        continue
    img  = X_test[idxs[0]]
    shap_images.append(img)
    shap_labels.append(cls_id)

shap_images_arr = np.array(shap_images)
shap_values     = explainer.shap_values(shap_images_arr)  # shape: [n_classes, n_samples, H, W, C]
print(f"SHAP values computed for {len(shap_images)} samples")

SHAP GradientExplainer ready


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor
Received: inputs=['Tensor(shape=(9, 32, 32, 3))']
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor
Received: inputs=['Tensor(shape=(50, 32, 32, 3))']
  warnings.warn(msg)


SHAP values computed for 9 samples


In [21]:
# --- Plot SHAP: for each sample show Original | SHAP heatmap | Overlay ---

n_samples = len(shap_images)
fig, axes = plt.subplots(n_samples, 3, figsize=(9, n_samples * 2.8))
if n_samples == 1:
    axes = axes.reshape(1, 3)

col_titles = ["Original", "SHAP Attribution\n(predicted class)", "SHAP Overlay"]
for col_idx, title in enumerate(col_titles):
    axes[0, col_idx].set_title(title, fontsize=10, fontweight='bold')

for row, (img, cls_id) in enumerate(zip(shap_images, shap_labels)):
    pred_cls = int(np.argmax(cnn_model.predict(img[np.newaxis], verbose=0)[0]))

    # SHAP values for the predicted class
    sv = shap_values[pred_cls][row]           # (H, W, C)
    sv_magnitude = np.sum(np.abs(sv), axis=2) # (H, W) — sum over channels
    sv_norm      = (sv_magnitude - sv_magnitude.min()) / \
                   (sv_magnitude.max() - sv_magnitude.min() + 1e-9)

    overlay = overlay_gradcam(img, sv_norm, alpha=0.5)

    # Col 0: original
    axes[row, 0].imshow(img)
    axes[row, 0].set_ylabel(
        f"True:{CLASS_NAMES[cls_id][:12]}\nPred:{CLASS_NAMES[pred_cls][:12]}",
        fontsize=6.5
    )
    axes[row, 0].axis('off')

    # Col 1: SHAP heatmap
    im = axes[row, 1].imshow(sv_norm, cmap='hot')
    axes[row, 1].axis('off')
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)

    # Col 2: overlay
    axes[row, 2].imshow(overlay)
    axes[row, 2].axis('off')

save_fig_to_pdf(fig, "SHAP Attributions — CNN (GradientExplainer, classes 0–8)")
print("SHAP page saved to PDF")

  → saved 'SHAP Attributions — CNN (GradientExplainer, classes 0–8)' to PDF
SHAP page saved to PDF


## 16. SHAP — Full 43-class Run (optional, slower)

In [22]:
# Uncomment to run SHAP for ALL 43 classes (produces multiple PDF pages)

# SHAP_ALL_CLASSES = list(range(43))
# shap_images_all, shap_labels_all = [], []
#
# for cls_id in SHAP_ALL_CLASSES:
#     idxs = np.where(y_test_raw == cls_id)[0]
#     if len(idxs) == 0:
#         continue
#     shap_images_all.append(X_test[idxs[0]])
#     shap_labels_all.append(cls_id)
#
# shap_arr_all  = np.array(shap_images_all)
# shap_vals_all = explainer.shap_values(shap_arr_all)
#
# BATCH = 9
# for page_i, start in enumerate(range(0, len(shap_images_all), BATCH)):
#     batch_imgs  = shap_images_all[start:start+BATCH]
#     batch_cls   = shap_labels_all[start:start+BATCH]
#     n = len(batch_imgs)
#     fig, axes = plt.subplots(n, 3, figsize=(9, n*2.8))
#     if n == 1: axes = axes.reshape(1,3)
#     for row, (img, cls_id) in enumerate(zip(batch_imgs, batch_cls)):
#         pred_cls = int(np.argmax(cnn_model.predict(img[np.newaxis], verbose=0)[0]))
#         sv = shap_vals_all[pred_cls][start+row]
#         sv_norm = np.sum(np.abs(sv), axis=2)
#         sv_norm = (sv_norm - sv_norm.min()) / (sv_norm.max() - sv_norm.min() + 1e-9)
#         overlay = overlay_gradcam(img, sv_norm, alpha=0.5)
#         axes[row,0].imshow(img);  axes[row,0].axis('off')
#         axes[row,0].set_ylabel(f"True:{CLASS_NAMES[cls_id][:12]}\nPred:{CLASS_NAMES[pred_cls][:12]}", fontsize=6)
#         axes[row,1].imshow(sv_norm, cmap='hot'); axes[row,1].axis('off')
#         axes[row,2].imshow(overlay); axes[row,2].axis('off')
#     save_fig_to_pdf(fig, f"SHAP All Classes — page {page_i+1}")

print("Full SHAP block skipped (uncomment above to enable)")

Full SHAP block skipped (uncomment above to enable)


## 17. Save Models

In [23]:
cnn_path = os.path.join(MODEL_DIR, "traffic_sign_cnn.h5")
mn_path  = os.path.join(MODEL_DIR, "traffic_sign_mobilenetv2.h5")

cnn_model.save(cnn_path)
mobilenet_model.save(mn_path)

print("CNN saved       :", cnn_path)
print("MobileNetV2 saved:", mn_path)

CNN saved       : /kaggle/working/outputs/models/traffic_sign_cnn.h5
MobileNetV2 saved: /kaggle/working/outputs/models/traffic_sign_mobilenetv2.h5


## 18. Finalise PDF  ⬅ MUST run last

In [24]:
# --- Attach PDF metadata ---
from datetime import datetime

pdf_meta = pdf.infodict()
pdf_meta['Title']   = 'Traffic Sign Recognition — Full Results'
pdf_meta['Author']  = 'Pattern Recognition Notebook'
pdf_meta['Subject'] = 'CNN vs MobileNetV2 | Grad-CAM | SHAP'
pdf_meta['CreationDate'] = datetime.now()

pdf.close()  # <-- finalises and flushes the file

size_mb = os.path.getsize(PDF_PATH) / 1e6
print(f"\n✅ PDF finalised: {PDF_PATH}")
print(f"   Size: {size_mb:.2f} MB")

# List all CSVs
csv_files = os.listdir(CSV_DIR)
print(f"\n✅ {len(csv_files)} CSV files in {CSV_DIR}:")
for f in sorted(csv_files):
    print("  ", f)


✅ PDF finalised: /kaggle/working/outputs/Traffic_Sign_Recognition_Results.pdf
   Size: 0.38 MB

✅ 9 CSV files in /kaggle/working/outputs/csv:
   class_distribution.csv
   cnn_classification_report.csv
   cnn_per_class_accuracy.csv
   cnn_training_history.csv
   dataset_split.csv
   mobilenetv2_classification_report.csv
   mobilenetv2_per_class_accuracy.csv
   mobilenetv2_training_history.csv
   model_comparison.csv
